# Practice Lab: Linear Regression

Welcome to your first practice lab! In this lab, you will implement linear regression with one variable to predict profits for a restaurant franchise.


# Outline
- [ 1 - Setup and Packages ](#1)
- [ 2 - Linear regression with one variable ](#2)
  - [ 2.1 Problem Statement](#2.1)
  - [ 2.2  Dataset](#2.2)
  - [ 2.3 Refresher on linear regression](#2.3)
  - [ 2.4  Compute Cost](#2.4)
    - [ Exercise 1](#ex01)
  - [ 2.5 Gradient descent ](#2.5)
    - [ Exercise 2](#ex02)
  - [ 2.6 Learning parameters using batch gradient descent ](#2.6)
    - [ Exercise 3](#ex03)
  - [2.7 Experimenting with Gradient Descent Hyperparameters](#2.7)
    - [ Exercise 4](#ex04)
- [ 3 - Linear regression using Scikit-Learn ](#3)
    - [Exercise 5](#ex05)


<a name="1"></a>
## 1 -  Setup and Packages

Before starting, make sure to mount your Google Drive and navigate to the correct directory where this notebook is located. This ensures you can access the dataset and utility libraries.

**Step 1: Mount Google Drive**

Run the following code to mount your Google Drive:


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

**Step 2: Navigate to the Notebook's Directory**

Change to the directory where this notebook is located. This ensures you can access the dataset and utility files:

In [ ]:
cd "/content/drive/MyDrive/Colab Notebooks/" ##YOUR PATH

**Step 3: Import Required Packages**

Next, let's run the cell below to import all the packages that you will need during this assignment.
- [numpy](www.numpy.org) is the fundamental package for working with matrices in Python.
- [matplotlib](http://matplotlib.org) is a famous library to plot graphs in Python.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import copy
import math

<a name="2"></a>
# 2 -  Linear regression with one variable

<a name="2.1"></a>
## 2.1 -  Problem Statement and dataset

Suppose you are the CEO of a restaurant franchise and are considering different cities for opening a new restaurant.
- You would like to expand your business to cities that may give your restaurant higher profits.
- The chain already has restaurants in various cities and you have data for profits and populations from the cities.
- You also have data on cities that are candidates for a new restaurant.
    - For these cities, you have the city population.
    
##**Can you use the data to help you identify which cities may potentially give your business higher profits?**

<a name="2.2"></a>
## 2.2 -  Dataset
You will start by loading the dataset for this task.
- The `load_data()` function shown below loads the data into variables `x_train` and `y_train`
  - `x_train` is the population of a city
  - `y_train` is the profit of a restaurant in that city. A negative value for profit indicates a loss.   
  - Both `X_train` and `y_train` are numpy arrays.

In [ ]:
def load_data():
    data = np.loadtxt("data/ex1data1.txt", delimiter=',')
    X = data[:,0]
    y = data[:,1]
    return X, y

In [ ]:
# load the dataset
x_train, y_train = load_data()

#### View the variables
Before starting on any task, it is useful to get more familiar with your dataset.  
- A good place to start is to just print out each variable and see what it contains.

The code below prints the variable `x_train` and the type of the variable.

In [ ]:
# print x_train
print("Type of x_train:",type(x_train))
print("First ten elements of x_train are:\n", x_train[:10])

`x_train` is a numpy array that contains decimal values that are all greater than zero.
- These values represent the city population times 10000
- For example, 6.1101 means that the population for that city is 61101
  
Now, let's print `y_train`

In [ ]:
# print y_train
print("Type of y_train:",type(y_train))
print("First ten elements of y_train are:\n", y_train[:10])

Similarly, `y_train` is a numpy array that has decimal values, some negative, some positive.
- These represent your restaurant's average monthly profits in each city, in units of 100000 da.
  - For example, 17.592 represents 1759200 in average monthly profits for that city.
  - -2.6807 represents -268070 in average monthly loss for that city.

#### Check the dimensions of your variables

Another useful way to get familiar with your data is to view its dimensions.

Please print the shape of `x_train` and `y_train` and see how many training examples you have in your dataset.

In [ ]:
print ('The shape of x_train is:', x_train.shape)
print ('The shape of y_train is: ', y_train.shape)
print ('Number of training examples (m):', len(x_train))

The city population array has 97 data points, and the monthly average profits also has 97 data points. These are NumPy 1D arrays.

#### Visualize your data

It is often useful to understand the data by visualizing it.
- For this dataset, you can use a scatter plot to visualize the data, since it has only two properties to plot (profit and population).
- Many other problems that you will encounter in real life have more than two properties (for example, population, average household income, monthly profits, monthly sales).When you have more than two properties, you can still use a scatter plot to see the relationship between each pair of properties.


In [ ]:
# Create a scatter plot of the data. To change the markers to red "x",
# we used the 'marker' and 'c' parameters
plt.scatter(x_train, y_train, marker='x', c='r')

# Set the title
plt.title("Profits vs. Population per city")
# Set the y-axis label
plt.ylabel('Profit in 100000 da')
# Set the x-axis label
plt.xlabel('Population of City in 10000s')
plt.show()

Your goal is to build a linear regression model to fit this data.
- With this model, you can then input a new city's population, and have the model estimate your restaurant's potential monthly profits for that city.

<a name="2.3"></a>
## 2.3 - Refresher on linear regression

In this practice lab, you will fit the linear regression parameters $(w,b)$ to your dataset.
- The model function for linear regression, which is a function that maps from `x` (city population) to `y` (your restaurant's monthly profit for that city) is represented as
    $$f_{w,b}(x) = wx + b$$
    

- To train a linear regression model, you want to find the best $(w,b)$ parameters that fit your dataset.  

    - To compare how one choice of $(w,b)$ is better or worse than another choice, you can evaluate it with a cost function $J(w,b)$
      - $J$ is a function of $(w,b)$. That is, the value of the cost $J(w,b)$ depends on the value of $(w,b)$.
  
    - The choice of $(w,b)$ that fits your data the best is the one that has the smallest cost $J(w,b)$.


- To find the values $(w,b)$ that gets the smallest possible cost $J(w,b)$, you can use a method called **gradient descent**.
  - With each step of gradient descent, your parameters $(w,b)$ come closer to the optimal values that will achieve the lowest cost $J(w,b)$.
  

- The trained linear regression model can then take the input feature $x$ (city population) and output a prediction $f_{w,b}(x)$ (predicted monthly profit for a restaurant in that city).

<a name="2.4"></a>
## 2.4 - Compute Cost

Gradient descent involves repeated steps to adjust the value of your parameter $(w,b)$ to gradually get a smaller and smaller cost $J(w,b)$.
- At each step of gradient descent, it will be helpful for you to monitor your progress by computing the cost $J(w,b)$ as $(w,b)$ gets updated.
- In this section, you will implement a function to calculate $J(w,b)$ so that you can check the progress of your gradient descent implementation.

#### Cost function
For one variable, the cost function for linear regression $J(w,b)$ is defined as

$$J(w,b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2$$

- You can think of $f_{w,b}(x^{(i)})$ as the model's prediction of your restaurant's profit, as opposed to $y^{(i)}$, which is the actual profit that is recorded in the data.
- $m$ is the number of training examples in the dataset

#### Model prediction

- For linear regression with one variable, the prediction of the model $f_{w,b}$ for an example $x^{(i)}$ is representented as:

$$ f_{w,b}(x^{(i)}) = wx^{(i)} + b$$

This is the equation for a line, with an **intercept $b$ and a slope $w$**.

## **Implementation**



<a name="ex01"></a>
### Exercise 1

Complete the `compute_cost` below to:

* Iterate over the training examples, and for each example, compute:
    * The prediction of the model for that example
    $$
    f_{wb}(x^{(i)}) =  wx^{(i)} + b
    $$
   
    * The cost for that example  $$cost^{(i)} =  (f_{wb} - y^{(i)})^2$$
    

* Return the total cost over all examples
$$J(\mathbf{w},b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} cost^{(i)}$$
  * Here, $m$ is the number of training examples and $\sum$ is the summation operator

In [ ]:
def compute_cost(x, y, w, b):
    """
    Computes the cost function for linear regression.

    Args:
        x (ndarray): Shape (m,) Input to the model (Population of cities)
        y (ndarray): Shape (m,) Label (Actual profits for the cities)
        w, b (scalar): Parameters of the model

    Returns
        total_cost (float): The cost of using w,b as the parameters for linear regression
               to fit the data points in x and y
    """
    # number of training examples
    m = x.shape[0]

    # You need to return this variable correctly
    total_cost = 0

    ### START CODE HERE ###
    # INSTRUCTIONS: Compute the cost function for linear regression.
    #               Your implementation MUST use vectorized operations with NumPy.
    #               Avoid using explicit loops (e.g., for or while loops).
    #
    # HINT: Use np.dot() for vectorized matrix multiplication and np.sum() for summation.
    #       Remember to compute the mean squared error cost as (1/(2*m)) * sum((predictions - y)^2).
    #
    # Steps:
    # 1. Compute the predictions for all examples using vectorized operations.
    # 2. Compute the errors (difference between predictions and actual labels).
    # 3. Square the errors.
    # 4. Sum the squared errors and divide by (2*m) to compute the cost.


    ### END CODE HERE ###

    return total_cost

You can check if your implementation was correct by running the following test code:

In [ ]:
# Compute cost with some initial values for paramaters w, b
initial_w = 2
initial_b = 1

cost = compute_cost(x_train, y_train, initial_w, initial_b)
print(type(cost))
print(f'Cost at initial w: {cost:.3f}')

# Public tests
from public_tests import *
compute_cost_test(compute_cost)

**Expected Output**:
<table>
  <tr>
    <td> <b>Cost at initial w:<b> 75.203 </td>
  </tr>
</table>

<a name="2.5"></a>
## 2.5 - Gradient descent

In this section, you will implement the gradient for parameters $w, b$ for linear regression.

The gradient descent algorithm is:

$$\begin{align*}& \text{repeat until convergence:} \; \lbrace \newline \; & \phantom {0000} b := b -  \alpha \frac{\partial J(w,b)}{\partial b} \newline       \; & \phantom {0000} w := w -  \alpha \frac{\partial J(w,b)}{\partial w} \tag{1}  \; &
\newline & \rbrace\end{align*}$$

where, parameters $w, b$ are both updated simultaniously and where  
$$
\frac{\partial J(w,b)}{\partial b}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)}) \tag{2}
$$
$$
\frac{\partial J(w,b)}{\partial w}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) -y^{(i)})x^{(i)} \tag{3}
$$
* m is the number of training examples in the dataset

    
*  $f_{w,b}(x^{(i)})$ is the model's prediction, while $y^{(i)}$, is the target value


You will implement a function called `compute_gradient` which calculates $\frac{\partial J(w)}{\partial w}$, $\frac{\partial J(w)}{\partial b}$

<a name="ex02"></a>
### Exercise 2

Please complete the `compute_gradient` function to:

* Iterate over the training examples, and for each example, compute:
    * The prediction of the model for that example
    $$
    f_{wb}(x^{(i)}) =  wx^{(i)} + b
    $$
   
    * The gradient for the parameters $w, b$ from that example
        $$
        \frac{\partial J(w,b)}{\partial b}^{(i)}  =  (f_{w,b}(x^{(i)}) - y^{(i)})
        $$
        $$
        \frac{\partial J(w,b)}{\partial w}^{(i)}  =  (f_{w,b}(x^{(i)}) -y^{(i)})x^{(i)}
        $$
    

* Return the total gradient update from all the examples
    $$
    \frac{\partial J(w,b)}{\partial b}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} \frac{\partial J(w,b)}{\partial b}^{(i)}
    $$
    
    $$
    \frac{\partial J(w,b)}{\partial w}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} \frac{\partial J(w,b)}{\partial w}^{(i)}
    $$
  * Here, $m$ is the number of training examples and $\sum$ is the summation operator

If you get stuck, you can check out the hints presented after the cell below to help you with the implementation.

In [ ]:
def compute_gradient(x, y, w, b):
    """
    Computes the gradient for linear regression
    Args:
      x (ndarray): Shape (m,) Input to the model (Population of cities)
      y (ndarray): Shape (m,) Label (Actual profits for the cities)
      w, b (scalar): Parameters of the model
    Returns
      dj_dw (scalar): The gradient of the cost w.r.t. the parameters w
      dj_db (scalar): The gradient of the cost w.r.t. the parameter b
     """

    # Number of training examples
    m = x.shape[0]

    # You need to return the following variables correctly
    dj_dw = 0
    dj_db = 0

    ### START CODE HERE ###
    # INSTRUCTIONS: Compute the gradient for linear regression.
    #               Your implementation MUST use vectorized operations with NumPy.
    #               Avoid using explicit loops (e.g., for or while loops).
    #
    # HINT: Use np.dot() for vectorized matrix multiplication and np.sum() for summation.
    #       Remember to compute the gradients as:
    #       dj_dw = (1/m) * sum((predictions - y) * x)
    #       dj_db = (1/m) * sum(predictions - y)
    #
    # Steps:
    # 1. Compute the predictions for all examples using vectorized operations.
    # 2. Compute the errors (difference between predictions and actual labels).
    # 3. Compute dj_dw as the mean of (errors * x).
    # 4. Compute dj_db as the mean of errors.


    ### END CODE HERE ###

    return dj_dw, dj_db

Run the cells below to check your implementation of the `compute_gradient` function with two different initializations of the parameters $w$,$b$.

In [ ]:
# Compute and display gradient with w initialized to zeroes
initial_w = 0
initial_b = 0

tmp_dj_dw, tmp_dj_db = compute_gradient(x_train, y_train, initial_w, initial_b)
print('Gradient at initial w, b (zeros):', tmp_dj_dw, tmp_dj_db)

compute_gradient_test(compute_gradient)

**Expected Output**:
<table>
  <tr>
    <td> <b>Gradient at initial , b (zeros)<b></td>
    <td> -65.32884975 -5.83913505154639</td>
  </tr>
</table>

In [ ]:
# Compute and display cost and gradient with non-zero w
test_w = 0.2
test_b = 0.2
tmp_dj_dw, tmp_dj_db = compute_gradient(x_train, y_train, test_w, test_b)

print('Gradient at test w, b:', tmp_dj_dw, tmp_dj_db)

**Expected Output**:
<table>
  <tr>
    <td> <b>Gradient at test w<b></td>
    <td> -47.41610118 -4.007175051546391</td>
  </tr>
</table>

<a name="2.6"></a>
## 2.6 Learning parameters using batch gradient descent

You will now find the optimal parameters of a linear regression model by using batch gradient descent. Recall batch refers to running all the examples in one iteration.

<a name="ex03"></a>
### Exercise 3

*   **Complete the Code:** Inside the loop, calculate the gradients (dj_dw, dj_db) using the provided gradient_function.
*   **Update the parameters w and b** using the gradients and learning rate alpha. The gradient update rule is:

$$\begin{align*}& \; \newline \; & \phantom {0000} b := b -  \alpha \frac{\partial J(w,b)}{\partial b} \newline       \; & \phantom {0000} w := w -  \alpha \frac{\partial J(w,b)}{\partial w} \tag{1}  \; &
\newline \end{align*}$$

- **Use the gradient_function to compute the gradients.**

- A good way to verify that gradient descent is working correctly is to look
at the value of $J(w,b)$ and check that it is decreasing with each step.

- Assuming you have implemented the gradient and computed the cost correctly and you have an appropriate value for the learning rate alpha, $J(w,b)$ should never increase and should converge to a steady value by the end of the algorithm.

In [ ]:
def gradient_descent(x, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters):
    """
    Performs batch gradient descent to learn theta. Updates theta by taking
    num_iters gradient steps with learning rate alpha

    Args:
      x :    (ndarray): Shape (m,)
      y :    (ndarray): Shape (m,)
      w_in, b_in : (scalar) Initial values of parameters of the model
      cost_function: function to compute cost
      gradient_function: function to compute the gradient
      alpha : (float) Learning rate
      num_iters : (int) number of iterations to run gradient descent
    Returns
      w : (ndarray): Shape (1,) Updated values of parameters of the model after
          running gradient descent
      b : (scalar)                Updated value of parameter of the model after
          running gradient descent
    """

    # number of training examples
    m = len(x)

    # An array to store cost J and w's at each iteration — primarily for graphing later
    J_history = []
    w_history = []
    w = copy.deepcopy(w_in)  #avoid modifying global w within function
    b = b_in

    for i in range(num_iters):

         ### START STUDENT CODE ###
        # Calculate the gradient and update the parameters

        # Update Parameters using w, b, alpha, and gradients

        ### END STUDENT CODE ###

        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion
            cost =  cost_function(x, y, w, b)
            J_history.append(cost)

        # Print cost every at intervals 100
        if (i+1)% 100 == 0:
            w_history.append(w)
            print(f"Iteration {i+1:4}: Cost {float(J_history[-1]):8.2f}   ")

    return w, b, J_history, w_history #return w and J,w history for graphing

Now let's run the gradient descent algorithm above to learn the parameters for our dataset.

In [ ]:
# initialize fitting parameters. Recall that the shape of w is (n,)
initial_w = 0.
initial_b = 0.

# some gradient descent settings
iterations = 2500
alpha = 0.015

w,b,J_history,_ = gradient_descent(x_train ,y_train, initial_w, initial_b,
                     compute_cost, compute_gradient, alpha, iterations)
print("w,b found by gradient descent:", w, b)

**Expected Output**:
<table>
  <tr>
    <td> <b> w, b found by gradient descent<b></td>
    <td> 1.1925747649094678 -3.891213135806608</td>
  </tr>
</table>

Now, we visualize the cost history of the gradient descent algorithm to analyze its convergence behavior. The cost history (J_history) is a record of the cost function's value at each iteration of gradient descent.

The plt.plot(J_history) function creates a line plot of the cost values stored in J_history, where the X-axis represents the iteration number, showing the progression of the algorithm over time. The Y-axis represents the cost value, which is the value of the cost function $J(w,b)$ at each iteration.


The plot helps us understand how well the gradient descent algorithm is performing:

- A steadily decreasing cost indicates that the algorithm is converging toward the optimal parameters.
- A plateau in the cost suggests that the algorithm has reached a minimum.
- Oscillations or an increasing cost may indicate issues with the learning rate or other hyperparameters.

## **Interpretation**

- If the cost decreases rapidly and then stabilizes, the learning rate (alpha) is likely well-chosen.
- If the cost decreases very slowly, the learning rate might be too small.
- If the cost fluctuates or diverges, the learning rate might be too large.

In [ ]:
# Plot the cost history
plt.plot(J_history)
plt.title(f"Cost History (alpha={alpha}, num_iters={iterations})")
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.show()

We will now use the final parameters from gradient descent to plot the linear fit.

Recall that we can get the prediction for a single example $f(x^{(i)})= wx^{(i)}+b$.

To calculate the predictions on the entire dataset, we can loop through all the training examples or use the vectorized implementation and calculate the prediction for each example. This is shown in the code block below.

In [ ]:
predicted = np.dot(x_train, w) + b

We will now plot the predicted values to see the linear fit.


In [ ]:
# Plot the linear fit
plt.plot(x_train, predicted, c = "b")

# Create a scatter plot of the data.
plt.scatter(x_train, y_train, marker='x', c='r')

# Set the title
plt.title("Profits vs. Population per city")
# Set the y-axis label
plt.ylabel('Profit in 1000000 da')
# Set the x-axis label
plt.xlabel('Population of City in 10000s')

Your final values of $w,b$ can also be used to make predictions on profits. Let's predict what the profit would be in areas of 35000 and 70000 people.

- The model takes in population of a city in 10,000s as input.

- Therefore, 35000 people can be translated into an input to the model as `np.array([3.5])`

- Similarly, 70000 people can be translated into an input to the model as `np.array([7.])`


In [ ]:
predict1 = 3.5 * w + b
print('For population = 35000, we predict a profit of %.2f' % (predict1*1000000))

predict2 = 7.0 * w + b
print('For population = 70000, we predict a profit of %.2f' % (predict2*1000000))

<a name="2.7"></a>
## 2.7 Experimenting with Gradient Descent Hyperparameters

In this section, you will experiment with different hyperparameters of gradient descent to observe their impact on the learning process. You will:

* Change the number of iterations.
* Adjust the learning rate (alpha).
* Modify the initial values of w and b.
* Observe how these changes affect the convergence of the algorithm and the final values of w and b.

<a name="ex04"></a>
### Exercise 4: Experimenting with Hyperparameters

1.  **Run the Gradient Descent Algorithm:**
    - Use the gradient_descent function implemented earlier.
    - Start with the default values: num_iters = 2500, alpha = 0.01, initial_w =initial_b = 0.

2.   **Experiment with Hyperparameters:**
    - **Change the number of iterations:** Try num_iters = 100, 1000, 2000,.... and observe how the cost changes over iterations and whether the algorithm converges.
    - **Adjust the learning rate:** Try alpha = 0.01, 0.015, 0.02.... and observe how the learning rate affects the speed of convergence and whether the algorithm diverges.
    - **Modify the initial values of w and b:** Try different starting points, such as initial_w = 10, initial_b = -5.  Observe how the initial values affect the convergence path.
    - **Visualize the Results:** Plot the cost history (J_history) for each experiment to visualize the convergence behavior.

3. **Answer the Following Questions:**
* What happens if the learning rate is too small? Too large?
* How does the number of iterations affect the final values of w and b?
* Do the initial values of w and b impact the final result? Why or why not?



In [ ]:
# Experiment with different hyperparameters
# Example: Change learning rate
alpha_values = [0.01, 0.015, 0.02]
for alpha in alpha_values:
    w, b, J_history, _ = gradient_descent(x_train, y_train, 0, 0, compute_cost, compute_gradient, alpha, 1000)
    plt.plot(J_history, label=f"alpha={alpha}")

plt.title("Cost History for Different Learning Rates")
plt.xlabel("Iteration")
plt.ylabel("Cost")
plt.legend()
plt.show()

<a name="3"></a>
## 3 -  Linear regresion using Scikit-Learn

There is an open-source, commercially usable machine learning toolkit called [scikit-learn](https://scikit-learn.org/stable/index.html). This toolkit contains implementations of many of the algorithms that you will work with in this course.


**Step 1: Import Required Packages**

In [ ]:
from sklearn.linear_model import LinearRegression

**Step 2: Create and fit the model**

The code below performs regression using scikit-learn. Scikit-learn has the [linear regression model](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LinearRegression.html#sklearn.linear_model.LinearRegression) which implements a closed-form linear regression.

<a name="ex05"></a>
### Exercise 5

Complete the code below to perform linear regression using Scikit-Learn. Follow these steps:

1.   **Create a regression object:** Use the appropriate class from Scikit-Learn to
create a linear regression object.
2.   **Fit the model:** Utilize the fit method associated with the regression object to perform regression. Note that the input data X must be a two-dimensional matrix (even if it has only one feature).

In [ ]:
### START CODE HERE ###
    # INSTRUCTIONS: Perform linear regression using Scikit-Learn.
    #
    # Steps:
    # 1. Create a regression object named linear_model: Use the appropriate class from Scikit-Learn to create a linear regression object.
    # 2. Fit the model: Utilize the fit method associated with the regression object to perform regression. Note that the input data X must be a two-dimensional matrix (even if it has only one feature).

### END CODE HERE ###


**Step 3: View Parameters**

The $\mathbf{w}$ and $\mathbf{b}$ parameters are referred to as 'coefficients' and 'intercept' in scikit-learn.

In [ ]:
b_sklearn = linear_model.intercept_
w_sklearn = linear_model.coef_
print(f"w = {w_sklearn[0]}, b = {b_sklearn}")

**Expected Output**:
<table>
  <tr>
    <td> <b> w, b found by Scikit-Learn<b></td>
    <td> 1.1930336441895932 -3.8957808783118484</td>
  </tr>
</table>

**Step 4: Make Predictions**

Calling the `predict` function generates predictions.

In [ ]:
predicted_s = linear_model.predict(x_train.reshape(-1, 1))

**Step 5: Plot the linear fit**

We will now plot the predicted values to see the linear fit and compare it with our model learned using Gradient Descent


In [ ]:
# Create a figure with two subplots side by side
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))  # 1 row, 2 columns

# Plot 1: Scikit-Learn Linear Fit
ax1.plot(x_train, predicted_s, c="m", label="Scikit-Learn Fit")  # Plot Scikit-Learn predictions
ax1.scatter(x_train, y_train, marker='x', c='r', label="Data Points")  # Scatter plot of data
ax1.set_title("Scikit-Learn Linear Fit")  # Set title
ax1.set_ylabel('Profit in 1000000 da')  # Set y-axis label
ax1.set_xlabel('Population of City in 10000s')  # Set x-axis label
ax1.legend()  # Add legend

# Plot 2: Your Implementation Linear Fit
ax2.plot(x_train, predicted, c="b", label="Your Implementation Fit")  # Plot your predictions
ax2.scatter(x_train, y_train, marker='x', c='r', label="Data Points")  # Scatter plot of data
ax2.set_title("Your Implementation Linear Fit")  # Set title
ax2.set_ylabel('Profit in 1000000 da')  # Set y-axis label
ax2.set_xlabel('Population of City in 10000s')  # Set x-axis label
ax2.legend()  # Add legend

# Display the plots
plt.tight_layout()  # Adjust layout to prevent overlap
plt.show()